In [ ]:
!git clone https://github.com/shreyans-gits/shape-e-model-generator
import sys
import os

sys.path.append('/content/shape-e-model-generator')
os.makedirs('/content/outputs', exist_ok=True)
!pip install -q git+https://github.com/openai/shap-e
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

fatal: destination path 'shape-e-model-generator' already exists and is not an empty directory.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/l

In [ ]:
import os
import sys

REPO_DIR = "/content/shape-e-model-generator"
REPO_URL = "https://github.com/shreyans-gits/shape-e-model-generator.git"

if os.path.exists(REPO_DIR):
    print("[Colab] Repository folder detected. Pulling changes from GitHub...")
    %cd {REPO_DIR}
    !git reset --hard HEAD
    !git pull origin main
else:
    print("[Colab] Repository not found. Cloning fresh copy...")
    !git clone {REPO_URL}
    %cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.append(src_dir)

print("[Colab] Ensuring all dependencies are installed...")
!pip install -q git+https://github.com/openai/shap-e
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("\n✓ Workspace sync complete. Ready for next cell!")

[Colab] Repository folder detected. Pulling changes from GitHub...
/content/shape-e-model-generator
HEAD is now at 2c11098 feat(drive): add generic uploader and in-memory reader for Colab
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 966 bytes | 966.00 KiB/s, done.
From https://github.com/shreyans-gits/shape-e-model-generator
 * branch            main       -> FETCH_HEAD
   2c11098..02a12ab  main       -> origin/main
Updating 2c11098..02a12ab
Fast-forward
 .gitignore      |  4 +++-
 drive_client.py | 40 +++++++++++++++++++++++++++++++++-------
 2 files changed, 36 insertions(+), 8 deletions(-)
[Colab] Ensuring all dependencies are installed...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done

✓ Workspace sync complete. Ready for next cell!


In [ ]:
import json
import drive_client
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google.colab import userdata

raw_token = userdata.get('GDRIVE_TOKEN')
token_info = json.loads(raw_token)

creds = Credentials.from_authorized_user_info(token_info, drive_client.SCOPES)
service = build('drive', 'v3', credentials=creds)

print("Authenticated with Google Drive via User OAuth Token!")

Authenticated with Google Drive via User OAuth Token!


In [ ]:
import torch
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading Shap-E models into memory (this takes 1-2 minutes)...")
xm = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))
print("Models loaded and ready for generation.")

Using device: cuda
Loading Shap-E models into memory (this takes 1-2 minutes)...
Models loaded and ready for generation.


In [ ]:
from src import sampler
from src import exporter
import drive_client

def process_job(job_dict, request_file_id, drive_service):
    job_id = job_dict.get('job_id')
    prompt = job_dict.get('prompt')
    fmt = job_dict.get('format', 'obj').lower()

    print(f"\n[Processing] Starting Job: {job_id} | Format: {fmt}")
    print(f"Prompt: \"{prompt}\"")
    latents = sampler.sample(
        prompt=prompt,
        model=model,
        diffusion=diffusion,
        batch_size=1,
        guidance_scale=3.0,
        steps=64
    )

    base_output_path = f"/content/outputs/{job_id}"

    exporter.export(
        latents=latents,
        xm=xm,
        output_path=base_output_path,
        format=fmt
    )
    generated_filename = f"{job_id}_0.{fmt}"
    local_file_path = f"/content/outputs/{generated_filename}"

    drive_client.upload_file(
        service=drive_service,
        file_path=local_file_path,
        filename=f"{job_id}.{fmt}",
        folder_id=drive_client.PENDING_FOLDER_ID
    )

    drive_client.delete_file(drive_service, request_file_id)
    print(f"[Colab] Job {job_id} handling complete. Cloud storage is clean.")

In [ ]:
import json
import time
import drive_client

CHECK_INTERVAL = 15

print("[Watcher] Starting continuous job watcher loop...")
print(f"[Watcher] Monitoring folder ID: {drive_client.PENDING_FOLDER_ID}")
print("-" * 60)

try:
    while True:
        files = drive_client.list_files(service, drive_client.PENDING_FOLDER_ID)
        json_found = False

        for file in files:
            filename = file.get('name', '')
            file_id = file.get('id')

            if filename.endswith('.json'):
                json_found = True
                print(f"\n[Watcher] Detected new job file: {filename}")

                try:
                    raw_json_str = drive_client.read_file_contents(service, file_id)
                    job_dict = json.loads(raw_json_str)
                    process_job(job_dict, file_id, service)

                except Exception as job_err:
                    print(f"[Watcher Error] Failed to process {filename}: {job_err}")
                    print("Skipping to next asset to protect loop integrity...")

        if not json_found:
            current_time = time.strftime("%H:%M:%S", time.localtime())
            print(f"Scanning pending/ channel... Clean at {current_time} ", end='\r')

        time.sleep(CHECK_INTERVAL)

except KeyboardInterrupt:
    print("\n[Watcher] Loop manually stopped by user. Exiting gracefully.")

[Watcher] Starting continuous job watcher loop...
[Watcher] Monitoring folder ID: 1ZGTtXVY-zXiDuzFV6XEJvdBLxKHmzyM0
------------------------------------------------------------

[Watcher] Detected new job file: 790eacf0.json

[Processing] Starting Job: 790eacf0 | Format: obj
Prompt: "a wooden chair"


  0%|          | 0/64 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/shap_e/models/stf/renderer.py:286: UserWarning: exception rendering with PyTorch3D: No module named 'pytorch3d'
  warnings.warn(f"exception rendering with PyTorch3D: {exc}")
/usr/local/lib/python3.12/dist-packages/shap_e/models/stf/renderer.py:287: UserWarning: falling back on native PyTorch renderer, which does not support full gradients
  warnings.warn(


Exported obj mesh to /content/outputs/790eacf0_0.obj
[Colab] Uploaded asset: 790eacf0.obj (ID: 1h7f0157oZNBZ_hiwW8okuW2VfaRSUgMP)
[Cleanup] Deleted remote file ID: 1bWr5VkY_1qL4BZpkNrAdErHorBLVjASY from Google Drive.
[Colab] Job 790eacf0 handling complete. Cloud storage is clean.

[Watcher] Detected new job file: 3dbceb18.json

[Processing] Starting Job: 3dbceb18 | Format: obj
Prompt: "a computer mouse"


  0%|          | 0/64 [00:00<?, ?it/s]

Exported obj mesh to /content/outputs/3dbceb18_0.obj
[Colab] Uploaded asset: 3dbceb18.obj (ID: 1tyPnw9XjdmcAu_KtX5FFl0piw7ucgG4Q)
[Cleanup] Deleted remote file ID: 1A4o9qbBLgQsPohpFUivVJFvOTu4Vyx5E from Google Drive.
[Colab] Job 3dbceb18 handling complete. Cloud storage is clean.

[Watcher] Detected new job file: 0540bca3.json

[Processing] Starting Job: 0540bca3 | Format: obj
Prompt: "a printer"


  0%|          | 0/64 [00:00<?, ?it/s]

Exported obj mesh to /content/outputs/0540bca3_0.obj
[Colab] Uploaded asset: 0540bca3.obj (ID: 1IYXh5hsONEv6UR-UXFF3cckWkTAnalGG)
[Cleanup] Deleted remote file ID: 1IC0cvakfKF5azsd-L2zIQ_egO44yQgaJ from Google Drive.
[Colab] Job 0540bca3 handling complete. Cloud storage is clean.

[Watcher] Detected new job file: e8bf0066.json

[Processing] Starting Job: e8bf0066 | Format: obj
Prompt: "a chips packet"


  0%|          | 0/64 [00:00<?, ?it/s]

Exported obj mesh to /content/outputs/e8bf0066_0.obj
[Colab] Uploaded asset: e8bf0066.obj (ID: 1OFLh45L7AOMfBYw2-XwiWAO8RlKztPA4)
[Cleanup] Deleted remote file ID: 1Dhk8Sgh_j4neYoafmSsSMrkw4DkDf4yT from Google Drive.
[Colab] Job e8bf0066 handling complete. Cloud storage is clean.

[Watcher] Loop manually stopped by user. Exiting gracefully.
